In [12]:
from cgra import *
from kernels import *
from scripts import sat_to_csv

In [13]:
kernel_name = "benchmarks/compigra/mmul_compi"
version = "_50x80x70"

In [14]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [15]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [16]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    #   -         -           &inputX[0]  &inputY[0]
    #   -         -           -           &output[0] 

    # int inputX[NI*NK], int inputY[NK*NJ], int output[NI*NJ]

    first_addr_A = first_addr
    first_addr_B = first_addr_A + rowsA*colsA*4
    first_addr_C = first_addr_B + colsA*colsB*4
    config_vals_col0 = []
    config_vals_col1 = []
    config_vals_col2 = [first_addr_A]
    config_vals_col3 = [first_addr_B, first_addr_C]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [17]:
def runKernel(load_addrs, max_it=1000, printVal=1):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it, printVal=printVal)

In [18]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [19]:
def gemm_cpu(A_data, B_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum
    return expected_res

In [20]:
# Test dimensions
rowsA = 50
colsA = 80
colsB = 70
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
C_data = [0 for _ in range(0, rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
C_data_cpy = C_data.copy()

#print("A")
#printAsMatrix(A_data, rowsA, colsA)
#print("B")
#printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB)

In [21]:
runKernel(load_addrs, max_it=20000000,printVal=0)



END


In [22]:
# Get result from CGRA
first_addr_C = first_addr + rowsA*colsA*4 + colsA*colsB*4
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)


# Get cpu output
expected_res = gemm_cpu(A_data_cpy, B_data_cpy, rowsA, colsA, colsB)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("CGRA: ")
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")



OK
